# Analysis of Marker Embeddings Evolution (stream_id_A and stream_id_B)

This notebook analyzes how marker embeddings (stream_id_A and stream_id_B) change during training.

**Goal**: Track the dynamics of marker embeddings changes across saved checkpoints.


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from tqdm.auto import tqdm
import pandas as pd

# Настройка графиков
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Увеличим размер графиков по умолчанию
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12


## 1. Loading Configuration and Checkpoint List


In [ ]:
# Путь к директории с чекпоинтами
RUN_DIR = Path('/disk/4tb/tikhonov/projects/mixture-of-embeddings/checkpoints/run_20251221_112940')

# Загрузка конфигурации
with open(RUN_DIR / 'training_config.json', 'r') as f:
    config = json.load(f)

print("Training Configuration:")
print("=" * 80)
for key, value in config.items():
    print(f"{key}: {value}")
print("=" * 80)


In [ ]:
# Найти все чекпоинты
checkpoint_dirs = sorted([d for d in RUN_DIR.iterdir() if d.is_dir() and d.name.startswith('checkpoint-')],
                         key=lambda x: int(x.name.split('-')[1]))

checkpoint_steps = [int(d.name.split('-')[1]) for d in checkpoint_dirs]

print(f"\nFound checkpoints: {len(checkpoint_dirs)}")
print(f"Steps: {checkpoint_steps}")


## 2. Extracting Marker Embeddings from Checkpoints


In [ ]:
def load_marker_embeddings(checkpoint_path):
    """
    Loads marker embeddings stream_id_A and stream_id_B from checkpoint.
    
    Returns:
        tuple: (stream_id_A, stream_id_B) as numpy arrays
    """
    checkpoint = torch.load(checkpoint_path / 'model.pt', map_location='cpu')
    
    # Extract marker embeddings
    # They have shape [1, 1, hidden_dim]
    stream_id_A = checkpoint['stream_id_A'].squeeze().numpy()  # [hidden_dim]
    stream_id_B = checkpoint['stream_id_B'].squeeze().numpy()  # [hidden_dim]
    
    return stream_id_A, stream_id_B


In [ ]:
# Загружаем векторы-маркеры из всех чекпоинтов
markers_A = []
markers_B = []

print("Loading marker embeddings from checkpoints...")
for checkpoint_dir in tqdm(checkpoint_dirs):
    stream_id_A, stream_id_B = load_marker_embeddings(checkpoint_dir)
    markers_A.append(stream_id_A)
    markers_B.append(stream_id_B)

# Convert to numpy arrays for convenience
# Shape: [num_checkpoints, hidden_dim]
markers_A = np.array(markers_A)
markers_B = np.array(markers_B)

print(f"\nShape of markers_A: {markers_A.shape}")
print(f"Shape of markers_B: {markers_B.shape}")


## 3. Analysis of Marker Evolution Metrics


In [ ]:
def compute_metrics(markers_A, markers_B):
    """
    Computes various metrics for marker embeddings.
    
    Returns:
        dict: dictionary with metrics
    """
    num_checkpoints = len(markers_A)
    
    metrics = {
        'norm_A': [],
        'norm_B': [],
        'cosine_sim_AB': [],  # cosine similarity between A and B
        'euclidean_dist_AB': [],  # euclidean distance between A and B
        'norm_diff_AB': [],  # norm difference
        'mean_A': [],  # mean component value
        'mean_B': [],
        'std_A': [],  # component standard deviation
        'std_B': [],
    }
    
    for i in range(num_checkpoints):
        marker_A = markers_A[i]
        marker_B = markers_B[i]
        
        # Norms
        norm_A = np.linalg.norm(marker_A)
        norm_B = np.linalg.norm(marker_B)
        metrics['norm_A'].append(norm_A)
        metrics['norm_B'].append(norm_B)
        
        # Cosine similarity
        cosine_sim = np.dot(marker_A, marker_B) / (norm_A * norm_B + 1e-8)
        metrics['cosine_sim_AB'].append(cosine_sim)
        
        # Euclidean distance
        euclidean_dist = np.linalg.norm(marker_A - marker_B)
        metrics['euclidean_dist_AB'].append(euclidean_dist)
        
        # Norm difference
        norm_diff = abs(norm_A - norm_B)
        metrics['norm_diff_AB'].append(norm_diff)
        
        # Component statistics
        metrics['mean_A'].append(np.mean(marker_A))
        metrics['mean_B'].append(np.mean(marker_B))
        metrics['std_A'].append(np.std(marker_A))
        metrics['std_B'].append(np.std(marker_B))
    
    return metrics

# Вычисляем метрики
metrics = compute_metrics(markers_A, markers_B)

# Создаем DataFrame для удобства
df_metrics = pd.DataFrame(metrics)
df_metrics['step'] = checkpoint_steps
df_metrics = df_metrics.set_index('step')

print("Metrics computed!")
df_metrics.head()


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(df_metrics.index, df_metrics['norm_A'], label='Norm(stream_id_A)', marker='o', linewidth=2)
ax.plot(df_metrics.index, df_metrics['norm_B'], label='Norm(stream_id_B)', marker='s', linewidth=2)

ax.set_xlabel('Training Step', fontsize=14)
ax.set_ylabel('L2 Norm', fontsize=14)
ax.set_title('Evolution of Marker Embedding Norms', fontsize=16, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Statistics of norm changes
print("\nNorm statistics:")
print(f"stream_id_A - start: {df_metrics['norm_A'].iloc[0]:.4f}, end: {df_metrics['norm_A'].iloc[-1]:.4f}, change: {df_metrics['norm_A'].iloc[-1] - df_metrics['norm_A'].iloc[0]:.4f}")
print(f"stream_id_B - start: {df_metrics['norm_B'].iloc[0]:.4f}, end: {df_metrics['norm_B'].iloc[-1]:.4f}, change: {df_metrics['norm_B'].iloc[-1] - df_metrics['norm_B'].iloc[0]:.4f}")


## 5. Analysis of Cosine Similarity Between Markers


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(df_metrics.index, df_metrics['cosine_sim_AB'], label='Cosine Similarity', 
        marker='o', linewidth=2, color='#e74c3c')

ax.set_xlabel('Training Step', fontsize=14)
ax.set_ylabel('Cosine Similarity', fontsize=14)
ax.set_title('Cosine Similarity between stream_id_A and stream_id_B', fontsize=16, fontweight='bold')
ax.axhline(y=0, color='k', linestyle='--', alpha=0.3)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Statistics
print("\nCosine similarity statistics:")
print(f"Start: {df_metrics['cosine_sim_AB'].iloc[0]:.4f}")
print(f"End: {df_metrics['cosine_sim_AB'].iloc[-1]:.4f}")
print(f"Change: {df_metrics['cosine_sim_AB'].iloc[-1] - df_metrics['cosine_sim_AB'].iloc[0]:.4f}")
print(f"Minimum: {df_metrics['cosine_sim_AB'].min():.4f} (step {df_metrics['cosine_sim_AB'].idxmin()})")
print(f"Maximum: {df_metrics['cosine_sim_AB'].max():.4f} (step {df_metrics['cosine_sim_AB'].idxmax()})")


## 6. Euclidean Distance Between Markers


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(df_metrics.index, df_metrics['euclidean_dist_AB'], label='Euclidean Distance', 
        marker='o', linewidth=2, color='#9b59b6')

ax.set_xlabel('Training Step', fontsize=14)
ax.set_ylabel('Euclidean Distance', fontsize=14)
ax.set_title('Euclidean Distance between stream_id_A and stream_id_B', fontsize=16, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Statistics
print("\nEuclidean distance statistics:")
print(f"Start: {df_metrics['euclidean_dist_AB'].iloc[0]:.4f}")
print(f"End: {df_metrics['euclidean_dist_AB'].iloc[-1]:.4f}")
print(f"Change: {df_metrics['euclidean_dist_AB'].iloc[-1] - df_metrics['euclidean_dist_AB'].iloc[0]:.4f}")


## 7. Marker Embedding Component Statistics


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Mean компонент
axes[0, 0].plot(df_metrics.index, df_metrics['mean_A'], label='Mean(stream_id_A)', marker='o', linewidth=2)
axes[0, 0].plot(df_metrics.index, df_metrics['mean_B'], label='Mean(stream_id_B)', marker='s', linewidth=2)
axes[0, 0].set_xlabel('Training Step')
axes[0, 0].set_ylabel('Mean Value')
axes[0, 0].set_title('Mean Component Values')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].axhline(y=0, color='k', linestyle='--', alpha=0.3)

# Std компонент
axes[0, 1].plot(df_metrics.index, df_metrics['std_A'], label='Std(stream_id_A)', marker='o', linewidth=2)
axes[0, 1].plot(df_metrics.index, df_metrics['std_B'], label='Std(stream_id_B)', marker='s', linewidth=2)
axes[0, 1].set_xlabel('Training Step')
axes[0, 1].set_ylabel('Standard Deviation')
axes[0, 1].set_title('Component Standard Deviation')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Разница норм
axes[1, 0].plot(df_metrics.index, df_metrics['norm_diff_AB'], label='|Norm(A) - Norm(B)|', 
                marker='o', linewidth=2, color='#f39c12')
axes[1, 0].set_xlabel('Training Step')
axes[1, 0].set_ylabel('Norm Difference')
axes[1, 0].set_title('Norm Difference Between Markers')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Сводная информация - траектория в пространстве (Norm A vs Norm B)
axes[1, 1].scatter(df_metrics['norm_A'], df_metrics['norm_B'], 
                   c=df_metrics.index, cmap='viridis', s=100, alpha=0.7)
axes[1, 1].plot(df_metrics['norm_A'], df_metrics['norm_B'], 
                alpha=0.3, linestyle='--', color='gray')
axes[1, 1].set_xlabel('Norm(stream_id_A)')
axes[1, 1].set_ylabel('Norm(stream_id_B)')
axes[1, 1].set_title('Norm Trajectory in Space')
axes[1, 1].grid(True, alpha=0.3)
# Add diagonal for reference
max_norm = max(df_metrics['norm_A'].max(), df_metrics['norm_B'].max())
min_norm = min(df_metrics['norm_A'].min(), df_metrics['norm_B'].min())
axes[1, 1].plot([min_norm, max_norm], [min_norm, max_norm], 'k--', alpha=0.3, label='y=x')
axes[1, 1].legend()

# Colorbar для последнего графика
cbar = plt.colorbar(axes[1, 1].collections[0], ax=axes[1, 1])
cbar.set_label('Training Step', rotation=270, labelpad=20)

plt.tight_layout()
plt.show()


In [ ]:
# Compute change of each component from first to last checkpoint
delta_A = markers_A[-1] - markers_A[0]
delta_B = markers_B[-1] - markers_B[0]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Распределение изменений компонент для stream_id_A
axes[0, 0].hist(delta_A, bins=50, alpha=0.7, color='#3498db', edgecolor='black')
axes[0, 0].set_xlabel('Δ Component Value')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of Component Changes (stream_id_A)')
axes[0, 0].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[0, 0].grid(True, alpha=0.3)

# Распределение изменений компонент для stream_id_B
axes[0, 1].hist(delta_B, bins=50, alpha=0.7, color='#2ecc71', edgecolor='black')
axes[0, 1].set_xlabel('Δ Component Value')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Distribution of Component Changes (stream_id_B)')
axes[0, 1].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[0, 1].grid(True, alpha=0.3)

# Топ-10 компонент с наибольшим изменением (stream_id_A)
top_indices_A = np.argsort(np.abs(delta_A))[-10:]
axes[1, 0].barh(range(10), delta_A[top_indices_A], color='#3498db')
axes[1, 0].set_yticks(range(10))
axes[1, 0].set_yticklabels([f'Comp {i}' for i in top_indices_A])
axes[1, 0].set_xlabel('Δ Value')
axes[1, 0].set_title('Top-10 Components with Largest Change (stream_id_A)')
axes[1, 0].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[1, 0].grid(True, alpha=0.3, axis='x')

# Топ-10 компонент с наибольшим изменением (stream_id_B)
top_indices_B = np.argsort(np.abs(delta_B))[-10:]
axes[1, 1].barh(range(10), delta_B[top_indices_B], color='#2ecc71')
axes[1, 1].set_yticks(range(10))
axes[1, 1].set_yticklabels([f'Comp {i}' for i in top_indices_B])
axes[1, 1].set_xlabel('Δ Value')
axes[1, 1].set_title('Топ-10 компонент с наибольшим изменением (stream_id_B)')
axes[1, 1].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[1, 1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print(f"\nСтатистика изменений компонент:")
print(f"stream_id_A:")
print(f"  Среднее изменение: {np.mean(delta_A):.6f}")
print(f"  Std изменения: {np.std(delta_A):.6f}")
print(f"  Макс увеличение: {np.max(delta_A):.6f}")
print(f"  Макс уменьшение: {np.min(delta_A):.6f}")
print(f"\nstream_id_B:")
print(f"  Среднее изменение: {np.mean(delta_B):.6f}")
print(f"  Std изменения: {np.std(delta_B):.6f}")
print(f"  Макс увеличение: {np.max(delta_B):.6f}")
print(f"  Макс уменьшение: {np.min(delta_B):.6f}")


In [ ]:
# Выберем 50 компонент с наибольшей вариацией во времени
def select_most_varying_components(markers, n_components=50):
    """Выбирает n компонент с наибольшей вариацией во времени."""
    variances = np.var(markers, axis=0)
    top_indices = np.argsort(variances)[-n_components:]
    return top_indices

# Выбираем компоненты
top_varying_A = select_most_varying_components(markers_A, n_components=50)
top_varying_B = select_most_varying_components(markers_B, n_components=50)

fig, axes = plt.subplots(1, 2, figsize=(18, 10))

# Тепловая карта для stream_id_A
im1 = axes[0].imshow(markers_A[:, top_varying_A].T, aspect='auto', cmap='RdBu_r', 
                     interpolation='nearest')
axes[0].set_xlabel('Checkpoint Index', fontsize=12)
axes[0].set_ylabel('Component Index', fontsize=12)
axes[0].set_title('Эволюция топ-50 varying компонент (stream_id_A)', fontsize=14, fontweight='bold')
axes[0].set_xticks(range(len(checkpoint_steps)))
axes[0].set_xticklabels([f'{step//1000}K' for step in checkpoint_steps], rotation=45)
plt.colorbar(im1, ax=axes[0], label='Component Value')

# Тепловая карта для stream_id_B
im2 = axes[1].imshow(markers_B[:, top_varying_B].T, aspect='auto', cmap='RdBu_r', 
                     interpolation='nearest')
axes[1].set_xlabel('Checkpoint Index', fontsize=12)
axes[1].set_ylabel('Component Index', fontsize=12)
axes[1].set_title('Эволюция топ-50 varying компонент (stream_id_B)', fontsize=14, fontweight='bold')
axes[1].set_xticks(range(len(checkpoint_steps)))
axes[1].set_xticklabels([f'{step//1000}K' for step in checkpoint_steps], rotation=45)
plt.colorbar(im2, ax=axes[1], label='Component Value')

plt.tight_layout()
plt.show()


## 10. Анализ скорости изменения маркеров

Вычислим и визуализируем скорость изменения векторов-маркеров между последовательными чекпоинтами.


In [ ]:
# Вычислим разницу между последовательными чекпоинтами
deltas_A = np.diff(markers_A, axis=0)
deltas_B = np.diff(markers_B, axis=0)

# Вычислим норму изменений (скорость изменения)
change_rate_A = np.linalg.norm(deltas_A, axis=1)
change_rate_B = np.linalg.norm(deltas_B, axis=1)

# Середины интервалов для графика
step_midpoints = [(checkpoint_steps[i] + checkpoint_steps[i+1]) / 2 
                  for i in range(len(checkpoint_steps) - 1)]

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Скорость изменения
axes[0].plot(step_midpoints, change_rate_A, label='Change Rate (stream_id_A)', 
             marker='o', linewidth=2, color='#3498db')
axes[0].plot(step_midpoints, change_rate_B, label='Change Rate (stream_id_B)', 
             marker='s', linewidth=2, color='#2ecc71')
axes[0].set_xlabel('Training Step', fontsize=12)
axes[0].set_ylabel('L2 Norm of Change', fontsize=12)
axes[0].set_title('Скорость изменения векторов-маркеров между чекпоинтами', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Логарифмическая шкала для лучшей визуализации
axes[1].semilogy(step_midpoints, change_rate_A, label='Change Rate (stream_id_A)', 
                 marker='o', linewidth=2, color='#3498db')
axes[1].semilogy(step_midpoints, change_rate_B, label='Change Rate (stream_id_B)', 
                 marker='s', linewidth=2, color='#2ecc71')
axes[1].set_xlabel('Training Step', fontsize=12)
axes[1].set_ylabel('L2 Norm of Change (log scale)', fontsize=12)
axes[1].set_title('Скорость изменения (логарифмическая шкала)', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nСтатистика скорости изменений:")
print(f"stream_id_A:")
print(f"  Средняя скорость: {np.mean(change_rate_A):.6f}")
print(f"  Std скорости: {np.std(change_rate_A):.6f}")
print(f"  Максимальная скорость: {np.max(change_rate_A):.6f} (между шагами {checkpoint_steps[np.argmax(change_rate_A)]} и {checkpoint_steps[np.argmax(change_rate_A)+1]})")
print(f"  Минимальная скорость: {np.min(change_rate_A):.6f} (между шагами {checkpoint_steps[np.argmin(change_rate_A)]} и {checkpoint_steps[np.argmin(change_rate_A)+1]})")
print(f"\nstream_id_B:")
print(f"  Средняя скорость: {np.mean(change_rate_B):.6f}")
print(f"  Std скорости: {np.std(change_rate_B):.6f}")
print(f"  Максимальная скорость: {np.max(change_rate_B):.6f} (между шагами {checkpoint_steps[np.argmax(change_rate_B)]} и {checkpoint_steps[np.argmax(change_rate_B)+1]})")
print(f"  Минимальная скорость: {np.min(change_rate_B):.6f} (между шагами {checkpoint_steps[np.argmin(change_rate_B)]} и {checkpoint_steps[np.argmin(change_rate_B)+1]})")


## 11. PCA анализ траектории маркеров

Проанализируем траекторию маркеров в пространстве главных компонент.


In [ ]:
from sklearn.decomposition import PCA

# Объединим маркеры для совместного PCA
markers_combined = np.vstack([markers_A, markers_B])

# Применим PCA
pca = PCA(n_components=2)
markers_pca = pca.fit_transform(markers_combined)

# Разделим обратно на A и B
markers_A_pca = markers_pca[:len(markers_A)]
markers_B_pca = markers_pca[len(markers_A):]

fig, ax = plt.subplots(figsize=(12, 10))

# Траектория stream_id_A
scatter_A = ax.scatter(markers_A_pca[:, 0], markers_A_pca[:, 1], 
                       c=checkpoint_steps, cmap='Blues', s=150, alpha=0.7,
                       marker='o', edgecolors='black', linewidths=1.5, label='stream_id_A')
ax.plot(markers_A_pca[:, 0], markers_A_pca[:, 1], 
        alpha=0.3, linestyle='-', color='blue', linewidth=2)

# Траектория stream_id_B
scatter_B = ax.scatter(markers_B_pca[:, 0], markers_B_pca[:, 1], 
                       c=checkpoint_steps, cmap='Greens', s=150, alpha=0.7,
                       marker='s', edgecolors='black', linewidths=1.5, label='stream_id_B')
ax.plot(markers_B_pca[:, 0], markers_B_pca[:, 1], 
        alpha=0.3, linestyle='-', color='green', linewidth=2)

# Отметим начальную и конечную точки
ax.scatter(markers_A_pca[0, 0], markers_A_pca[0, 1], 
           s=300, marker='*', color='red', edgecolors='black', linewidths=2, 
           label='Start A', zorder=5)
ax.scatter(markers_A_pca[-1, 0], markers_A_pca[-1, 1], 
           s=300, marker='P', color='darkred', edgecolors='black', linewidths=2, 
           label='End A', zorder=5)
ax.scatter(markers_B_pca[0, 0], markers_B_pca[0, 1], 
           s=300, marker='*', color='lime', edgecolors='black', linewidths=2, 
           label='Start B', zorder=5)
ax.scatter(markers_B_pca[-1, 0], markers_B_pca[-1, 1], 
           s=300, marker='P', color='darkgreen', edgecolors='black', linewidths=2, 
           label='End B', zorder=5)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}% variance)', fontsize=14)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}% variance)', fontsize=14)
ax.set_title('Траектория векторов-маркеров в пространстве главных компонент', 
             fontsize=16, fontweight='bold')
ax.legend(fontsize=11, loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nОбъяснённая дисперсия первых 2 PC: {pca.explained_variance_ratio_[:2].sum()*100:.2f}%")
print(f"PC1: {pca.explained_variance_ratio_[0]*100:.2f}%")
print(f"PC2: {pca.explained_variance_ratio_[1]*100:.2f}%")


In [ ]:
# Создадим сводную таблицу с ключевыми метриками
summary = {
    'Метрика': [
        'Norm(stream_id_A) - начало',
        'Norm(stream_id_A) - конец',
        'Norm(stream_id_A) - изменение',
        'Norm(stream_id_B) - начало',
        'Norm(stream_id_B) - конец',
        'Norm(stream_id_B) - изменение',
        'Cosine Similarity - начало',
        'Cosine Similarity - конец',
        'Cosine Similarity - изменение',
        'Euclidean Distance - начало',
        'Euclidean Distance - конец',
        'Euclidean Distance - изменение',
    ],
    'Значение': [
        df_metrics['norm_A'].iloc[0],
        df_metrics['norm_A'].iloc[-1],
        df_metrics['norm_A'].iloc[-1] - df_metrics['norm_A'].iloc[0],
        df_metrics['norm_B'].iloc[0],
        df_metrics['norm_B'].iloc[-1],
        df_metrics['norm_B'].iloc[-1] - df_metrics['norm_B'].iloc[0],
        df_metrics['cosine_sim_AB'].iloc[0],
        df_metrics['cosine_sim_AB'].iloc[-1],
        df_metrics['cosine_sim_AB'].iloc[-1] - df_metrics['cosine_sim_AB'].iloc[0],
        df_metrics['euclidean_dist_AB'].iloc[0],
        df_metrics['euclidean_dist_AB'].iloc[-1],
        df_metrics['euclidean_dist_AB'].iloc[-1] - df_metrics['euclidean_dist_AB'].iloc[0],
    ]
}

df_summary = pd.DataFrame(summary)
df_summary['Значение'] = df_summary['Значение'].apply(lambda x: f'{x:.6f}')

print("\n" + "="*80)
print("СВОДНАЯ ТАБЛИЦА МЕТРИК")
print("="*80)
print(df_summary.to_string(index=False))
print("="*80)


## 13. Экспорт данных для дальнейшего анализа


In [ ]:
# Сохраним все данные для дальнейшего анализа
output_dir = RUN_DIR / 'marker_analysis'
output_dir.mkdir(exist_ok=True)

# Сохраним метрики
df_metrics.to_csv(output_dir / 'marker_metrics.csv')
print(f"Метрики сохранены в: {output_dir / 'marker_metrics.csv'}")

# Сохраним сырые данные маркеров
np.save(output_dir / 'markers_A.npy', markers_A)
np.save(output_dir / 'markers_B.npy', markers_B)
print(f"Сырые данные маркеров сохранены в: {output_dir}")

# Сохраним checkpoint_steps
np.save(output_dir / 'checkpoint_steps.npy', np.array(checkpoint_steps))
print(f"Шаги чекпоинтов сохранены в: {output_dir / 'checkpoint_steps.npy'}")

print("\nВсе данные успешно экспортированы!")
